In [4]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_parquet("clean_paragraphs.parquet")
texts = df["TEXT_SPAN"].tolist()


In [6]:
df


,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name
0,81893,81893_p0,La commission propose de ne pas donner suite à...,President,FR,Bugnon André
1,81894,81894_p0,"Ich möchte Sie nur nochmals bitten, das Result...",For the committee,DE,Widmer Hans
2,81894,81894_p1,"In diesem Sinne bitte ich Sie nochmals, den An...",For the committee,DE,Widmer Hans
3,81895,81895_p0,Der Bundesrat beantragt die Annahme des Postul...,Member of the federal council,DE,Schmid Samuel
4,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel
...,...,...,...,...,...,...
303379,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine
303380,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine
303381,288681,288681_p13,En ce qui concerne le cas précis de cette prop...,For the committee,FR,Python Valentine
303382,288681,288681_p14,La majorité de la commission désire prendre en...,For the committee,FR,Python Valentine


In [7]:
embeddings = np.load("paragraph_embeddings.npy")
embeddings.shape


(303384, 1024)

In [8]:
embeddings = embeddings / np.linalg.norm(
    embeddings, axis=1, keepdims=True
)  # Normalize embeddings to unit length


In [9]:
from sklearn.cluster import MiniBatchKMeans

print(f"Total embeddings shape: {embeddings.shape}")

n_neighborhoods = 100
print(f"Clustering into {n_neighborhoods} semantic neighborhoods...")

kmeans = MiniBatchKMeans(
    n_clusters=n_neighborhoods, batch_size=1024, random_state=42, n_init="auto", verbose=1
)
neighborhood_labels = kmeans.fit_predict(embeddings)

df["Semantic_Neighborhood"] = neighborhood_labels

print("Extracting stratified samples...")
diverse_sample_df = (
    df.groupby("Semantic_Neighborhood", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 10), random_state=42))
    .copy()
    .reset_index(drop=True)
)

print(f"Successfully extracted {len(diverse_sample_df)} paragraphs!")

Total embeddings shape: (303384, 1024)
Clustering into 100 semantic neighborhoods...
Init 1/1 with method k-means++
Inertia for init 1/1: 2369.98095703125
Minibatch step 1/29627: mean batch inertia: 0.7723168730735779
[MiniBatchKMeans] Reassigning 4 cluster centers.
Minibatch step 2/29627: mean batch inertia: 0.5059073567390442, ewa inertia: 0.5059073567390442
Minibatch step 3/29627: mean batch inertia: 0.49634790420532227, ewa inertia: 0.5058428256686581
Minibatch step 4/29627: mean batch inertia: 0.4900623559951782, ewa inertia: 0.5057362996311437
[MiniBatchKMeans] Reassigning 8 cluster centers.
Minibatch step 5/29627: mean batch inertia: 0.4898313283920288, ewa inertia: 0.5056289331459921
Minibatch step 6/29627: mean batch inertia: 0.4776109457015991, ewa inertia: 0.5054397977626142
Minibatch step 7/29627: mean batch inertia: 0.47807443141937256, ewa inertia: 0.5052550678970276
Minibatch step 8/29627: mean batch inertia: 0.4764966368675232, ewa inertia: 0.5050609341503083
Minibatch 

/tmp/ipykernel_6706/2802072279.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 10), random_state=42))


In [10]:
diverse_sample_df.to_parquet("1k_diverse_paragraphs_for_LLM.parquet", index=False)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

# Model
model_id = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [26]:
import llm

SYSTEM_PROMPT = """You are an expert political scientist.
Extract the core policy claim from the text as a concise English sentence.
Output MUST be in English.
If the text is procedural (voting, greetings, etc.), output exactly: NONE."""

qwen_model = llm.get_model("sdsc-qwen-32b")

for idx, row in tqdm(diverse_sample_df.iterrows(), total=len(diverse_sample_df)):
    PROMPT = row["TEXT_SPAN"]
    response = qwen_model.prompt(prompt=PROMPT, system=SYSTEM_PROMPT, temperature=0.0)
    answer = response.text().split("</think>")[-1]
    print(answer)

  0%|          | 0/965 [00:00<?, ?it/s]



We should focus on ongoing reforms, avoid new projects, and reject the Recordon postulate.


This postulate should be rejected.


The UDC group rejects the Landolt proposal.


KeyboardInterrupt: 

In [ ]:
claims_df = diverse_sample_df[
    (diverse_sample_df["Extracted_Claim"].str.upper() != "NONE")
    & (diverse_sample_df["Extracted_Claim"].str.len() > 10)
].copy()


In [ ]:
print(claims_df[["TEXT_SPAN", "Extracted_Claim"]].head(10))
